# Stylized Facts of Equity Growth Rates
Investors gain insights into market behavior, risk, and opportunity by examining the continuously compounded growth rates $g^{(i)}_t$ of equities and their empirical properties. These properties are called **stylized facts**. We focus on three that matter most for portfolio construction and risk:

* __Heavy (fat) tailed distributions:__ Equity growth rates exhibit tails heavier than a normal distribution. Extreme moves are rarer than the bulk, but much more likely than a Gaussian would predict.
* __Absence of linear autocorrelation:__ The autocorrelation of $g^{(i)}_t$ at positive lags is close to zero, consistent with a random-walk view in which past growth rates do not predict future growth rates.
* __Volatility clustering:__ The autocorrelation of $|g^{(i)}_t|$ (or $(g^{(i)}_t)^2$) is strongly positive for many lags. Large moves cluster in time, small moves cluster in time, and volatility persists.

These three motivate the course-wide choice of a **hybrid JumpHMM generator**: regime-switching dynamics capture clustered volatility, demeaned HMM marginals reproduce heavy tails, and a Student-t copula couples the names. A Gaussian SIM alone would miss all three facts.

For deeper reading:
* Cont, R. (2001). Empirical properties of asset returns: stylized facts and statistical issues. *Quantitative Finance*, 1(2), 223-236. [Link](https://doi.org/10.1080/713665670)
* Mandelbrot, B. (1963). The Variation of Certain Speculative Prices. *The Journal of Business*, 36(4), 394-419. [Link](http://www.jstor.org/stable/2350970)

The companion [StylizedFacts example notebook](../eCornell-AI-Finance-S1-Example-StylizedFacts-Example-May-2026.ipynb) fits Hill estimators and plots ACFs on real data; this notebook develops the math only.

___

## Learning Objectives
* __Heavy tails and the Hill estimator:__ You will write the heavy-tail definition in terms of a regularly varying tail function and identify the tail index $\alpha$. You will apply Hill's estimator to a sample of order statistics.
* __Sample autocorrelation of growth rates:__ You will compute the biased sample autocovariance and autocorrelation function of $g^{(i)}_t$. You will connect a flat ACF to the random-walk view of prices.
* __Volatility clustering via ACF of $|g|$:__ You will compute the ACF of absolute or squared growth rates and see slow decay over many lags. You will explain why Gaussian SIM models fail this fact and motivate the hybrid generator used later in the course.

___

## Heavy Tailed Return Distributions
Heavy‐tailed distributions have probability of extreme deviations that decays more slowly than exponentially, making large shocks rarer but much more likely than under a normal distribution. 

> __Black swan events__: Heavy tails in stock‐return distributions make extreme jumps far more frequent than normal distributions predict. These _black swans_, such as the 1987 crash, 2008 crisis, the COVID-19 pandemic, or April 2025 tariff sell-offs, arise from larger systemic failures, or sudden news, earnings surprises, or geopolitical shocks.

A **heavy-tailed distribution** has tails that are not exponentially bounded. For random variable $X$, tail probabilities decay more slowly than exponential:
$$
\mathbb{P}(|X| > x) \sim L(x)\,x^{-\alpha}, \quad \text{as } x \to \infty,
$$
where $\alpha > 0$ is the **tail index** (controlling decay rate) and $L(x)$ is a **slowly varying function** at infinity: $\lim_{x \to \infty} \frac{L(tx)}{L(x)} = 1$ for all $t > 0$.

> __Tail index__: 
> The tail index $\alpha$ is a key parameter in characterizing the heaviness of the tail. It determines how quickly the tail probabilities decay. 
> A smaller $\alpha$ indicates a heavier tail, while a larger $\alpha$ indicates a lighter tail. Here's a guide:
> * If $\alpha \leq 1$, the distribution is **very heavy-tailed** (e.g., Cauchy has $\alpha = 1$, meaning the mean does not exist).
> * If $1 < \alpha < 2$, the distribution has an **infinite variance** but a finite mean.
> * If $2 \leq \alpha \leq 4$, the distribution is **heavy-tailed** with finite variance (typical of equity returns).
> * If $\alpha > 4$, the distribution is often treated as **effectively light-tailed**.

#### Hill's estimator of the tail index
Hill's estimator is a method for estimating the tail index $\alpha$ of a heavy-tailed distribution. 
Let $X_1, X_2, \dots, X_n$ be i.i.d. realizations of a heavy-tailed random variable $X$, and let $X_{(1)} \ge X_{(2)} \ge \cdots \ge X_{(n)}$ denote the **order statistics** (sorted in decreasing order). Fix $k < n$, the number of **upper order statistics** to use (typically with $k \ll n$), then the **Hill estimator** of the **tail index $\alpha$** is:
$$
\boxed{
\hat{\alpha}_k = \left[ \frac{1}{k} \sum_{i=1}^{k} \ln\left(\frac{X_{(i)}}{X_{(k+1)}}\right) \right]^{-1}.
}
$$

The choice of $k$ is crucial, as it balances bias and variance in the estimate; we'll typically look at a range of $k$ values and plot the estimates to find a stable region.
___

## Autocorrelation
Autocorrelation is a key concept in time series analysis, particularly in the context of financial returns. Suppose we have the growth rate series $\left\{g_2^{(i)},g_3^{(i)},\ldots,g_T^{(i)}\right\}$ for firm $i$. The (empirical) autocovariance at lag $k$ (where we neglect the firm superscript for simplicity) is given by the **biased estimator**:
$$
\hat{R}(k) = \frac{1}{T-1}\sum_{t=2}^{T-k}\left(g_t - \bar{g}\right)\left(g_{t+k} - \bar{g}\right)\quad\;k=0,1,\ldots,T-2
$$
where $k$ is the lag (units: time steps) and $\bar{g}$ is the mean of the continuously compounded growth rates over the $T - 1$ samples. Note: we divide by $T-1$ (the total number of observations) for all lags, not by $T-1-k$. This biased convention ensures that the autocovariance matrix is positive semi-definite. The autocorrelation function (ACF) is the normalized autocovariance:
$$
\boxed{
\rho(k) = \frac{\hat{R}(k)}{\hat{R}(0)}\quad\;k=0,1,\ldots,T-2
}
$$
The ACF has some nice properties: $|\rho(k)| \leq 1$ for all $k$, and $\rho(0) = 1$ by definition. Additionally, for a stationary process, we expect the ACF to decay to zero as $k$ increases, indicating that past returns have less influence on future returns over time.

> __Random Walk:__ In the famous book [A Random Walk Down Wall Street](https://en.wikipedia.org/wiki/A_Random_Walk_Down_Wall_Street), Burton Malkiel argued that stock prices follow a random walk, meaning that past price movements do not predict future price movements. In this case, the autocorrelation function (ACF) of stock returns would be close to zero for all lags $k > 0$. This suggests that stock prices are unpredictable and follow a random walk.

If the random walk hypothesis holds, then the autocorrelation function (ACF) of the continuously compounded growth rates $\left\{g_2^{(i)},g_3^{(i)},\ldots,g_T^{(i)}\right\}$ should be close to zero for all lags $k > 0$ and all firms $i$. In other words, there should be no significant correlation between past and future returns.

___

## Volatility Clustering
Volatility clustering refers to the tendency for large (in absolute value) returns to be followed by large returns, and small returns by small ones, creating persistent periods of high and low volatility.

> __Why volatility clustering matters:__ Dynamic risk must be modeled to avoid mis-estimated risk measures (like Value at Risk), derivative prices, and portfolio allocations. Incorporating clustering improves risk forecasts, option valuations, hedging strategies, and capital allocation decisions.

A standard proxy for volatility at time $t$ is the absolute return $|g_t^{(i)}|$ (we can also use squared returns $(g_t^{(i)})^2$; both capture the same clustering effect). Let firm $i$ have growth rate series: $\bigl\{g_{2}^{(i)},\,g_{3}^{(i)},\,\dots,\,g_{T}^{(i)}\bigr\},$
and let the sample mean of the squared growth rates be given by:
$$
\overline{(g^2)_{i}} 
\;=\;\frac{1}{T-1}\sum_{t=2}^T\bigl(g_t^{(i)}\bigr)^2
$$
The normalized empirical autocorrelation of squared returns at lag $k$ is then:
$$
\boxed{
\hat{\rho}_i(k)
=\frac{\displaystyle
   \overbrace{\sum_{t=2}^{\,T-k}
     \Bigl[\bigl(g_t^{(i)}\bigr)^2-\overline{(g^2)_{i}}\Bigr]
     \,\Bigl[\bigl(g_{t+k}^{(i)}\bigr)^2-\overline{(g^2)_{i}}\Bigr]
   }^{\hat R_i(k)\;\text{with squared growth rates}}
}{\displaystyle
   \underbrace{\sum_{t=2}^{\,T}
     \Bigl[\bigl(g_t^{(i)}\bigr)^2-\overline{(g^2)_{i}}\Bigr]^{2}
   }_{\hat R_i(0)\;\text{with squared growth rates}}
}\,.}
$$

This is equivalent to the biased ACF convention from the previous section: dividing both $\hat{R}_i(k)$ and $\hat{R}_i(0)$ by the same factor $\frac{1}{T-1}$ cancels in the ratio, so we write the sums directly. Volatility clustering is observed when $\hat{\rho}_i(k)>0$ for many $k$ and typically $\hat{\rho}_i(k)\searrow0$ slowly as $k\to\infty$.

___

## Summary: Why These Three Facts Drive Course Design
The three stylized facts above are not independent curiosities. They jointly rule out the simplest plausible return model (Gaussian i.i.d.) and constrain what a credible scenario generator must do.

> __Design implications:__
>
> __Heavy tails__ rule out $\sigma$-based risk measures alone. The course uses empirical VaR/CVaR and empirical confidence cones from full wealth-path samples, not $\pm k\sigma$ normal bands. __Absence of autocorrelation in $g$__ justifies the SIM regression assumptions (uncorrelated residuals over time) and justifies the random-walk null in simple backtests. __Volatility clustering__ requires a time-varying variance model: this is why the hybrid JumpHMM generator is built on a regime-switching backbone, not an i.i.d. draw.

## Key Takeaways
* __Tail index $\alpha$ governs risk:__ Equity growth rates typically live in the $2 \leq \alpha \leq 4$ regime, giving finite variance but materially heavy tails. Hill's estimator gives an empirical $\hat{\alpha}_k$ whose stability across $k$ is itself diagnostic.
* __$g$ is unpredictable; $|g|$ is predictable:__ The raw growth-rate ACF sits near zero, but the absolute (or squared) growth-rate ACF decays slowly. Any generator that must survive backtesting has to reproduce this asymmetry.
* __Hybrid generator motivation:__ Regime-switching dynamics + heavy-tailed marginals + a Student-t copula is a minimal design that captures all three stylized facts simultaneously, which is why the course stress tests are built on it rather than on a Gaussian SIM.